In [1]:
import pandas as pd

human = pd.read_csv("../data/region_exports/human_M1_S1_V1_metadata.csv", index_col="cell_label")
mouse = pd.read_csv("../data/region_exports/mouse_M1_S1_V1_metadata.csv", index_col="cell_label")

In [3]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

In [5]:
download_base = Path('../../data/abc_atlas')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

c:\Users\lilyg\miniforge3\envs\neuroscience490\Lib\site-packages\abc_atlas_access\abc_atlas_cache\cloud_cache.py:679: OutdatedManifestWarning: You are loading releases/20261031/manifest.json. A more up to date version of the dataset -- releases/20261130/manifest.json -- exists online. To see the changes between the two versions of the dataset, run
type.compare_manifests('releases/20261031/manifest.json', 'releases/20261130/manifest.json')
To load another version of the dataset, run
type.load_manifest('releases/20261130/manifest.json')
  warnings.warn(msg, OutdatedManifestWarning)


'releases/20261031/manifest.json'

In [6]:
import anndata as ad
import numpy as np
from scipy import sparse


def pull_raw_counts(pilot, directory, matrix, symbol, matrix_path=None):
    genes = abc_cache.get_metadata_dataframe(directory=directory, file_name="gene")
    gene_ids = genes.loc[genes["gene_symbol"].eq(symbol), "gene_identifier"]

    if matrix_path is None:
        matrix_path = abc_cache.get_data_path(
            directory=directory,
            file_name=f"{matrix}/raw",
        )

    data = ad.read_h5ad(matrix_path, backed="r")
    try:
        gene_columns = np.flatnonzero(data.var_names.isin([symbol, *gene_ids]))
        if len(gene_columns) != 1:
            raise ValueError(
                f"Expected one column for {symbol} in {matrix}, "
                f"found {len(gene_columns)}. Gene IDs from metadata: {gene_ids.tolist()}"
            )
        gene_column = int(gene_columns[0])

        # Match by cell_label, then read rows in matrix order.
        row_positions = data.obs_names.get_indexer(pilot.index)
        matched = row_positions >= 0
        sorted_rows = np.sort(row_positions[matched])

        counts = np.empty(len(sorted_rows), dtype=float)
        batch_size = 8192

        for start in range(0, len(sorted_rows), batch_size):
            stop = min(start + batch_size, len(sorted_rows))
            rows = sorted_rows[start:stop]
            block = data[rows, gene_column:gene_column + 1].X
            if sparse.issparse(block):
                block = block.toarray()
            counts[start:stop] = np.asarray(block).ravel()

        gene_counts = pd.Series(
            counts,
            index=data.obs_names[sorted_rows],
            name="raw_count",
        )
    finally:
        data.file.close()

    result = pilot.join(gene_counts, validate="one_to_one")
    print(f"{symbol}: matched {matched.sum():,} of {len(pilot):,} cells")
    print(f"Detected in {(result['raw_count'] > 0).sum():,} matched cells")
    return result

In [8]:
mouse_selected = mouse.loc[
    mouse["neurotransmitter"].isin(["GABA", "Glut"])
].copy()

mouse_htr2a = pull_raw_counts(
    mouse_selected,
    directory="Consensus-WMB-Macosko-10X",
    matrix="Macosko-10X-Isocortex",
    symbol="Htr2a",
)

mouse_htr2a.to_csv(
    "../data/region_exports/mouse_M1_S1_V1_Htr2a_counts.csv",
    index_label="cell_label",
)

Htr2a: matched 164,935 of 164,935 cells
Detected in 48,413 matched cells


In [9]:
mouse_detection = (
    mouse_htr2a.assign(detected=mouse_htr2a["raw_count"].gt(0))
    .groupby(["region", "donor_label", "neurotransmitter"])
    .agg(
        n_cells=("raw_count", "count"),
        n_detected=("detected", "sum"),
    )
)

mouse_detection["pct_detected"] = (
    100 * mouse_detection["n_detected"] / mouse_detection["n_cells"]
)

display(mouse_detection["pct_detected"].unstack("neurotransmitter").round(2))

neurotransmitter     GABA   Glut
region donor_label              
M1     F003         27.58  35.40
       F004         20.36  38.63
       F005         28.61  40.67
       F006         22.79  20.79
       F007         20.95  45.26
       F008         21.63  40.54
       M002         24.68  48.10
       M003         19.67  44.30
       M004         25.07  38.48
       M006         26.29  44.23
       M007         25.86  42.52
       M008         20.48  39.91
S1     F022         20.81  25.08
V1     F019         23.32  15.63
       M019         22.38  13.29

In [13]:
transmitter = human["neurotransmitter"]
gaba_only = transmitter.eq("GABA").fillna(False)
vglut_only = transmitter.str.fullmatch(r"VGLUT[123](?: VGLUT[123])*", na=False)

human_selected = human.loc[gaba_only | vglut_only].copy()
human_selected["neuron_group"] = "Glut"
human_selected.loc[human_selected["neurotransmitter"].eq("GABA"), "neuron_group"] = "GABA"

human_htr2a = pull_raw_counts(
    human_selected,
    directory="WHB-10Xv3",
    matrix="WHB-10Xv3-Neurons",
    symbol="HTR2A",
)

human_htr2a.to_csv(
    "../data/region_exports/human_M1_S1_V1_HTR2A_counts.csv",
    index_label="cell_label",
)

HTR2A: matched 167,462 of 167,462 cells
Detected in 92,265 matched cells


In [14]:
print(f"Exported cells: {len(human_htr2a):,}")
print(f"Unmatched cells: {human_htr2a['raw_count'].isna().sum():,}")

display(
    human_htr2a.loc[human_htr2a["region"].eq("M1")]
    .groupby(["supercluster", "donor_label"], dropna=False)
    .size()
    .unstack(fill_value=0)
)

matched = human_htr2a.dropna(subset=["raw_count"])
human_rates = (
    matched.assign(detected=matched["raw_count"].gt(0))
    .groupby(["region", "donor_label", "neuron_group"])["detected"]
    .mean()
    .mul(100)
    .unstack("neuron_group")
    .reindex(columns=["GABA", "Glut"])
)

human_rates["Glut − GABA (pp)"] = human_rates["Glut"] - human_rates["GABA"]
display(human_rates.round(2))

Exported cells: 167,462
Unmatched cells: 0


donor_label,H18.30.001,H18.30.002,H19.30.001,H19.30.002
supercluster,,,,
Amygdala excitatory,78,0,0,0
CGE interneuron,4352,4012,1736,1878
Deep-layer corticothalamic and 6b,3888,2161,665,518
Deep-layer intratelencephalic,11895,7286,1906,1787
Deep-layer near-projecting,1444,821,230,170
Eccentric medium spiny neuron,15,0,0,0
Hippocampal CA1-3,602,0,0,0
Hippocampal CA4,3,0,0,0
Hippocampal dentate gyrus,18,0,0,0


neuron_group         GABA   Glut  Glut − GABA (pp)
region donor_label                                
M1     H18.30.001   10.98  70.67             59.69
       H18.30.002   13.27  76.23             62.96
       H19.30.001    7.00  51.17             44.17
       H19.30.002    9.34  67.35             58.01
S1     H18.30.002   14.43  82.56             68.13
       H19.30.001    7.07  57.30             50.23
       H19.30.002    9.30  66.81             57.51
V1     H18.30.002   27.05  87.71             60.66
       H19.30.001    6.80  55.35             48.55
       H19.30.002   11.72  67.72             56.00